In [0]:
%run ../common/common_logging

In [0]:
%run ../common/common_config_loader

In [0]:
import uuid
from datetime import datetime, timezone, timedelta
import time
from decimal import Decimal
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    TimestampType, BooleanType, DecimalType, LongType
)

In [0]:
# Get configuration from common config loader
config = get_config()
logger = get_logger(__name__)

# Table References from config
BRONZE_API_LOG = config.get_bronze_table("bronze_api_response_log")
PIPELINE_RUN_CONTROL = f"{config.CATALOG}.metadata.pipeline_run_control"
AUDIT_PIPELINE_RUNS = f"{config.CATALOG}.audit.audit_pipeline_runs"

# Define explicit schemas for manifest tables

API_LOG_SCHEMA = StructType([
    StructField("response_log_id", StringType(), False),
    StructField("source_name", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("request_url", StringType(), False),
    StructField("http_status_code", IntegerType(), False),
    StructField("retry_count", IntegerType(), False),
    StructField("rate_limit_hit", BooleanType(), False),
    StructField("response_time_ms", IntegerType(), True),
    StructField("logged_at", TimestampType(), False)
])

PIPELINE_RUN_SCHEMA = StructType([
    StructField("run_control_sk", LongType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("source_name", StringType(), True),
    StructField("trigger_type", StringType(), False),
    StructField("scheduled_at", TimestampType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("status", StringType(), False),
    StructField("operator_user", StringType(), True)
])

AUDIT_RUN_SCHEMA = StructType([
    StructField("audit_run_sk", LongType(), False),
    StructField("run_id", StringType(), False),
    StructField("pipeline_name", StringType(), False),
    StructField("environment", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), False),
    StructField("rows_read", LongType(), True),
    StructField("rows_written", LongType(), True),
    StructField("runtime_seconds", DecimalType(18, 2), True),
    StructField("error_message", StringType(), True)
])

In [0]:
def log_api_response(source_name, batch_id, request_url, http_status_code, 
                     retry_count=0, rate_limit_hit=False, response_time_ms=None):
    """Log API response to bronze layer"""
    log_record = [{
        'response_log_id': str(uuid.uuid4()),
        'source_name': source_name,
        'batch_id': batch_id,
        'request_url': request_url,
        'http_status_code': http_status_code,
        'retry_count': retry_count,
        'rate_limit_hit': rate_limit_hit,
        'response_time_ms': response_time_ms,
        'logged_at': datetime.now(timezone.utc)
    }]
    
    df = spark.createDataFrame(log_record, schema=API_LOG_SCHEMA)
    df.write.mode('append').saveAsTable(BRONZE_API_LOG)
    
    # Log using common logger
    logger.debug(f"API response logged: {source_name} - Status {http_status_code} - {response_time_ms}ms")
    
    return log_record[0]['response_log_id']

In [0]:
def start_pipeline_run(pipeline_name, source_name, batch_id):
    """Register pipeline run start in metadata.pipeline_run_control"""
    run_data = [{
        'run_control_sk': int(time.time() * 1000),
        'pipeline_name': pipeline_name,
        'batch_id': batch_id,
        'source_name': source_name,
        'trigger_type': 'MANUAL',
        'scheduled_at': None,
        'started_at': datetime.now(timezone.utc),
        'ended_at': None,
        'status': 'RUNNING',
        'operator_user': None
    }]
    
    df = spark.createDataFrame(run_data, schema=PIPELINE_RUN_SCHEMA)
    df.write.mode('append').saveAsTable(PIPELINE_RUN_CONTROL)
    
    # Log using common logger
    logger.info(f"Started pipeline run: {pipeline_name} (batch: {batch_id})")
    
    return run_data[0]['run_control_sk']

def complete_pipeline_run(batch_id, status):
    """Update pipeline run status to completed"""
    spark.sql(f"""
        UPDATE {PIPELINE_RUN_CONTROL}
        SET status = '{status}',
            ended_at = current_timestamp()
        WHERE batch_id = '{batch_id}'
    """)
    
    # Log using common logger
    logger.info(f"Completed pipeline run (batch: {batch_id}) with status: {status}")

In [0]:
def log_audit_pipeline_run(run_id, pipeline_name, status, rows_read=0, rows_written=0, 
                          runtime_seconds=0.0, error_message=None):
    """Log comprehensive audit entry in audit.audit_pipeline_runs"""
    audit_data = [{
        'audit_run_sk': int(time.time() * 1000),
        'run_id': run_id,
        'pipeline_name': pipeline_name,
        'environment': 'PROD',
        'start_time': datetime.now(timezone.utc) - timedelta(seconds=runtime_seconds),
        'end_time': datetime.now(timezone.utc) if status != 'RUNNING' else None,
        'status': status,
        'rows_read': rows_read,
        'rows_written': rows_written,
        'runtime_seconds': Decimal(str(round(runtime_seconds, 2))),
        'error_message': error_message
    }]
    
    df = spark.createDataFrame(audit_data, schema=AUDIT_RUN_SCHEMA)
    df.write.mode('append').saveAsTable(AUDIT_PIPELINE_RUNS)
    
    # Log using common logger
    if status == 'SUCCESS':
        log_metrics(logger, {
            "Pipeline": pipeline_name,
            "Rows Read": rows_read,
            "Rows Written": rows_written,
            "Runtime (s)": round(runtime_seconds, 2)
        }, prefix="Audit")
    elif status == 'FAILED':
        logger.error(f"Pipeline {pipeline_name} failed: {error_message}")